In [ ]:
%pip install duckdb

# Análisis de Segmentación RFM y Optimización de Inventarios
**Contexto del Negocio:** Un retail de e-commerce requiere identificar sus segmentos de clientes más valiosos (VIP) frente a los que están en riesgo de abandono.
**Objetivo Analítico:** Calcular la Recencia, Frecuencia y Valor Monetario (RFM) para dirigir el presupuesto de marketing con precisión y no desperdiciarlo en correos masivos.

In [1]:
import duckdb
import pandas as pd

## 1. Ingestión de Datos Crudos (Raw Data)
Crearemos vistas virtuales directamente sobre nuestros archivos CSV inmutables en `/data/raw/` usando DuckDB para optimizar la memoria.

In [2]:
# Conectamos a una base de datos en memoria
con = duckdb.connect(database=':memory:')

# Registramos los CSVs como si fueran tablas de SQL
con.execute("CREATE VIEW customers AS SELECT * FROM read_csv_auto('../data/raw/olist_customers_dataset.csv')")
con.execute("CREATE VIEW orders AS SELECT * FROM read_csv_auto('../data/raw/olist_orders_dataset.csv')")
con.execute("CREATE VIEW order_items AS SELECT * FROM read_csv_auto('../data/raw/olist_order_items_dataset.csv')")

print("Vistas creadas exitosamente. Listo para consultar.")

Vistas creadas exitosamente. Listo para consultar.


## 2. Transformación Analítica (Cálculo RFM)
Ejecutaremos la lógica de negocio estructurada mediante CTEs y Funciones de Ventana para asignar cuartiles a cada cliente.

In [3]:
# Cargamos tu script SQL directamente desde la carpeta scripts
with open('../scripts/01_rfm_segmentation.sql', 'r') as file:
    rfm_sql_query = file.read()

# Ejecutamos la consulta y la guardamos en un DataFrame de Pandas
df_rfm = con.execute(rfm_sql_query).df()

# Mostramos las primeras 5 filas para verificar
display(df_rfm.head())

# GUARDAMOS EL RESULTADO EN LA CARPETA PROCESSED PARA POWER BI
df_rfm.to_csv('../data/processed/rfm_segments.csv', index=False)
print("¡ETL Finalizado! Datos guardados en /data/processed/rfm_segments.csv")

,customer_unique_id,recency_days,frequency,monetary,rfm_total_score,rfm_cell
0,d80730c15c647bc8f2ad77c908ba5ca9,127,1,0.85,8,341
1,b38211bd797f4fdd81a98b9d1754b606,126,1,0.85,8,341
2,317cfc692e3f86c45c95697c61c853a6,3,1,2.20,6,411
3,bd06ce0e06ad77a7f681f1a4960a3cc6,349,1,2.29,3,111
4,cf3839da0d9492ad151690b65f45d800,208,1,2.99,6,321


¡ETL Finalizado! Datos guardados en /data/processed/rfm_segments.csv


## 3. Segmentación Estratégica y Análisis de Negocio ("So What?")
Clasificaremos a los clientes en segmentos accionables basados en su matriz RFM para identificar oportunidades de retención y monetización.

In [4]:
import numpy as np

# Volvemos a leer el archivo procesado para trabajar en memoria con Pandas
df = pd.read_csv('../data/processed/rfm_segments.csv')

# Asegurarnos de que rfm_cell sea un string para facilitar el mapeo
df['rfm_cell'] = df['rfm_cell'].astype(str)

# Definimos una función de negocio para asignar segmentos reales
def assign_segment(rfm_cell):
    r, f, m = int(rfm_cell[0]), int(rfm_cell[1]), int(rfm_cell[2])
    
    # Lógica de segmentación avanzada
    if r == 4 and f >= 3 and m >= 3:
        return 'Champions (VIPs)'
    elif r >= 3 and f >= 3:
        return 'Leales (Loyal)'
    elif r >= 3 and f <= 2:
        return 'Nuevos y Prometedores'
    elif r <= 2 and f >= 3:
        return 'En Riesgo de Abandono (At Risk)'
    elif r <= 2 and f <= 2:
        return 'Perdidos (Hibernating/Lost)'
    else:
        return 'Atención Requerida'

# Aplicamos la función para crear la nueva columna
df['Customer_Segment'] = df['rfm_cell'].apply(assign_segment)

# Guardamos el dataset final enriquecido para Power BI
df.to_csv('../data/processed/rfm_segments_final.csv', index=False)

print("Segmentación aplicada con éxito.")
display(df['Customer_Segment'].value_counts())

Segmentación aplicada con éxito.


Customer_Segment
Perdidos (Hibernating/Lost)        27489
Leales (Loyal)                     19819
Nuevos y Prometedores              19191
En Riesgo de Abandono (At Risk)    19191
Champions (VIPs)                    7668
Name: count, dtype: int64

In [5]:
# Agrupamos por segmento para ver el impacto financiero
segment_analysis = df.groupby('Customer_Segment').agg(
    Total_Customers=('customer_unique_id', 'count'),
    Total_Revenue=('monetary', 'sum')
).reset_index()

# Calculamos el porcentaje de ingresos que representa cada segmento
total_revenue = segment_analysis['Total_Revenue'].sum()
segment_analysis['Revenue_Percentage'] = (segment_analysis['Total_Revenue'] / total_revenue) * 100

# Formateamos para que se vea como un reporte financiero
segment_analysis['Total_Revenue'] = segment_analysis['Total_Revenue'].apply(lambda x: f"${x:,.2f}")
segment_analysis['Revenue_Percentage'] = segment_analysis['Revenue_Percentage'].apply(lambda x: f"{x:.1f}%")

display(segment_analysis.sort_values(by='Total_Customers', ascending=False))

,Customer_Segment,Total_Customers,Total_Revenue,Revenue_Percentage
4,Perdidos (Hibernating/Lost),27489,"$3,888,190.03",29.4%
2,Leales (Loyal),19819,"$2,145,091.02",16.2%
1,En Riesgo de Abandono (At Risk),19191,"$2,688,790.57",20.3%
3,Nuevos y Prometedores,19191,"$2,630,382.91",19.9%
0,Champions (VIPs),7668,"$1,869,043.58",14.1%
